# day-27-observability — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [7]:
# ---- Solution 1 ----
def cost_rollup(root):
    total = dict(tok_in=0, tok_out=0, cost_usd=0.0)
    def walk(s):
        if s.kind == "llm":
            for kk in total: total[kk] += s.outputs.get(kk, 0)
        for c in s.children: walk(c)
    walk(root)
    root.metadata["cost_rollup"] = {k: round(v, 6) for k, v in total.items()}
    return total

_, tr = rag("how far ahead should I book travel")
r = cost_rollup(tr)
print(f"S1: request cost ${r['cost_usd']:.6f}  ({r['tok_in']} in / {r['tok_out']} out)")

S1: request cost $0.000130  (100 in / 6 out)


In [8]:
# ---- Solution 3 ----
def generate_strict(prompt, q, chunks):
    with span("generate", kind="llm") as s:
        if not prompt.strip(): raise ValueError("empty prompt")
        s.outputs = {"answer": "ok"}; return "ok"

with span("err_demo", kind="chain") as root:
    try:
        with span("sibling_ok") as sib: sib.outputs = {"fine": True}
        generate_strict("", "q", [])
    except ValueError:
        root.status = "error"
render(root)
print("S3: 'generate' and 'err_demo' show !! ; 'sibling_ok' stays ok.")

!! err_demo                  0.0ms  in={}  out={}
     sibling_ok                0.0ms  in={}  out={"fine": true}
  !! generate                  0.0ms  in={}  out={}
S3: 'generate' and 'err_demo' show !! ; 'sibling_ok' stays ok.


In [9]:
# ---- Solution 6 ----
def trace_to_testcase(root):
    ret = None
    def walk(s):
        nonlocal ret
        if s.name == "retrieve": ret = s
        for c in s.children: walk(c)
    walk(root)
    q = root.inputs["user_query"]
    return dict(id="prod_" + root.id, type="paraphrase" if "?" not in q else "single_fact",
                q=q, fact=None, source=ret.outputs["hits"][0]["doc"] if ret else None)

_, tr = rag("what's the hotel cap for London")
print("S6:", json.dumps(trace_to_testcase(tr), indent=1))
print("    (a human fills in the correct `fact` -- '$350' -- then it joins the eval set)")

S6: {
 "id": "prod_673a28bc",
 "type": "paraphrase",
 "q": "what's the hotel cap for London",
 "fact": null,
 "source": "travel"
}
    (a human fills in the correct `fact` -- '$350' -- then it joins the eval set)


### Solutions 2, 4, 5 (sketch)

**S2:** `by_name = defaultdict(list); for req in range(20): _, t = rag(q); for s in
flatten_objs(t): by_name[s.name].append(s.duration_ms)`. Then mean + `np.percentile(v, 95)`
per name. `generate` (the LLM call) is almost always the bottleneck; `retrieve` second.

**S4:** `re.sub(r"[\w.]+@[\w.]+", "[EMAIL]", text)`, `re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b",
"[PHONE]", ...)`, `re.sub(r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b", "[CARD]", ...)`. Apply
before assigning `s.inputs`. The point: redaction happens at the *instrumentation* layer so no
raw PII ever reaches the trace store.

**S5:** `store_full = random.random() < sample_rate`; in `build_prompt`,
`s.outputs["prompt"] = prompt if store_full else None`. Over 100 runs, `~sample_rate` spans
have non-None `prompt`. Always store full text for `status == "error"` regardless of the coin.

### Answer key
1. Logs (what happened at this moment — structured events); traces (the full path of one
   request — a tree of spans); metrics (aggregate health over time — numbers).
2. One request fans out into retrieval, reranking, tool calls, and generation; a wrong final
   answer can originate in any of them, and only the trace shows each stage's input and
   output so you can localise the fault.
3. A span is one timed operation with a name, start/end, inputs, outputs, metadata, status,
   and a parent. A trace is a root span plus all its descendants — one request.
4. Log: the query, k, retrieved ids + scores, index version. Don't log: the full chunk text
   on every request (store ids, fetch on demand) or raw PII.
5. Get the trace_id from the report/alert → open the trace, read top to bottom → find the
   first span whose output is wrong given a correct input → that span is the fault, fix it →
   add the failing input as a Day-26 test case → if it recurs, add a metric on that span.
6. Full-text logging of every request is a storage cost and a privacy/PII liability; sampling
   1–10% (plus 100% of errors/thumbs-downs) keeps debuggability while bounding both.
7. It's the `with span(...)` context manager wrapped around a function — it records the
   function's args as span inputs and its return value as span outputs, automatically,
   wherever the function is called.